# AAAI-13 Accepted Papers ETL

- **Source data**: `[UCI] AAAI-13 Accepted Papers - Papers.csv`
- **Pipeline**: Extract (load CSV) → Transform (cleaning/normalization) → Load (save output files)

Note: the source CSV is quoted multi-line format — fields such as Title/Keywords/Topics contain embedded line breaks.

In [ ]:
import re
import pandas as pd
from pathlib import Path

## 1. EXTRACT

In [13]:
CSV_PATH = Path(r'C:\Users\USER\Desktop\[16-08-26] Cloudy\03.AAAI-13 Accepted Papers\[UCI] AAAI-13 Accepted Papers - Papers.csv')
OUT_DIR = CSV_PATH.parent / 'output'
OUT_DIR.mkdir(exist_ok=True)

# Load CSV with quoted multi-line fields
df = pd.read_csv(CSV_PATH, engine='python')
print(df.shape)
df.head(3)

(150, 5)


,Title,Keywords,Topics,High-Level Keyword(s),Abstract
0,The cascade auction – a mechanism for deterrin...,Mediators\nAuctions\nCollusion\nAd Exchanges,Auctions and Market-Based Systems\nE-Commerce\...,Multiagent Systems,We introduce a sealed bid auction of a single ...
1,Basis Adaptation for Sparse Nonlinear Reinforc...,Reinforcement learning\nSparsity\nMirror desce...,Dimension Reduction/Feature Selection\nOnline ...,Machine Learning\nReasoning under Uncertainty,This paper presents a new approach to basis ad...
2,Optimal Coalition Structures in Cooperative Gr...,Cooperative Game Theory\nCoalition Structure G...,Coordination and Collaboration\nGame Theory,Multiagent Systems,Representation languages for coalitional game...


In [14]:
# Check missing values
df.isna().sum()

Title                    0
Keywords                 0
Topics                   0
High-Level Keyword(s)    0
Abstract                 0
dtype: int64

## 2. TRANSFORM

In [15]:
df_t = df.copy()

# 2-1. Remove duplicate rows
n_dup = df_t.duplicated().sum()
df_t = df_t.drop_duplicates().reset_index(drop=True)
print(f'duplicated rows removed: {n_dup}')

duplicated rows removed: 0


In [16]:
# 2-2. Normalize tag columns stored as multiple lines into a single comma-separated string
TAG_COLS = ['Keywords', 'Topics', 'High-Level Keyword(s)']

def normalize_tags(s):
    if pd.isna(s):
        return ''
    tags = [t.strip() for t in re.split(r'[\n;]+', str(s)) if t.strip()]
    return ', '.join(tags)

for col in TAG_COLS:
    df_t[col] = df_t[col].map(normalize_tags)
    df_t[f'{col}_count'] = df_t[col].map(lambda s: 0 if s == '' else len(s.split(', ')))

df_t[['Keywords', 'Topics', 'High-Level Keyword(s)']].head(3)

,Keywords,Topics,High-Level Keyword(s)
0,"Mediators, Auctions, Collusion, Ad Exchanges","Auctions and Market-Based Systems, E-Commerce,...",Multiagent Systems
1,"Reinforcement learning, Sparsity, Mirror desce...","Dimension Reduction/Feature Selection, Online ...","Machine Learning, Reasoning under Uncertainty"
2,"Cooperative Game Theory, Coalition Structure G...","Coordination and Collaboration, Game Theory",Multiagent Systems


In [17]:
# 2-3. Remove LaTeX markup from Title/Abstract (e.g. $...$, {\em ...})
def clean_latex(s):
    if pd.isna(s):
        return ''
    s = re.sub(r'\$([^$]*)\$', r'\1', str(s))
    s = re.sub(r'\{\\em\s+([^}]*)\}', r'\1', s)
    s = re.sub(r'\{\\textit\s+([^}]*)\}', r'\1', s)
    s = re.sub(r'\{\\bf\s+([^}]*)\}', r'\1', s)
    s = s.replace('\\', ' ')
    return s

df_t['Title'] = df_t['Title'].map(clean_latex)
df_t['Abstract'] = df_t['Abstract'].map(clean_latex)
df_t[['Title']].head(3)

,Title
0,The cascade auction – a mechanism for deterrin...
1,Basis Adaptation for Sparse Nonlinear Reinforc...
2,Optimal Coalition Structures in Cooperative Gr...


In [18]:
# 2-4. Normalize whitespace (consecutive spaces/tabs -> single space, internal line breaks -> space)
for col in df_t.columns:
    df_t[col] = df_t[col].map(
        lambda s: re.sub(r'\s+', ' ', s).strip() if isinstance(s, str) else s
    )
print(df_t.shape)
df_t.head(2)

(150, 8)


,Title,Keywords,Topics,High-Level Keyword(s),Abstract,Keywords_count,Topics_count,High-Level Keyword(s)_count
0,The cascade auction – a mechanism for deterrin...,"Mediators, Auctions, Collusion, Ad Exchanges","Auctions and Market-Based Systems, E-Commerce,...",Multiagent Systems,We introduce a sealed bid auction of a single ...,4,4,1
1,Basis Adaptation for Sparse Nonlinear Reinforc...,"Reinforcement learning, Sparsity, Mirror desce...","Dimension Reduction/Feature Selection, Online ...","Machine Learning, Reasoning under Uncertainty",This paper presents a new approach to basis ad...,5,4,2


In [19]:
# 2-5. Derived columns: abstract word count, title length
df_t['Abstract_word_count'] = df_t['Abstract'].map(lambda s: len(s.split()) if isinstance(s, str) else 0)
df_t['Title_length'] = df_t['Title'].map(lambda s: len(s) if isinstance(s, str) else 0)
df_t[['Title', 'Abstract_word_count', 'Title_length']].head(3)

,Title,Abstract_word_count,Title_length
0,The cascade auction – a mechanism for deterrin...,69,69
1,Basis Adaptation for Sparse Nonlinear Reinforc...,155,60
2,Optimal Coalition Structures in Cooperative Gr...,189,55


In [20]:
# 2-6. Summary of the cleaned dataset
print('columns:', list(df_t.columns))
print('nulls:')
print(df_t.isna().sum())
print('top topics:')
print(df_t['Topics'].str.split(', ').explode().value_counts().head(10))

columns: ['Title', 'Keywords', 'Topics', 'High-Level Keyword(s)', 'Abstract', 'Keywords_count', 'Topics_count', 'High-Level Keyword(s)_count', 'Abstract_word_count', 'Title_length']
nulls:
Title                          0
Keywords                       0
Topics                         0
High-Level Keyword(s)          0
Abstract                       0
Keywords_count                 0
Topics_count                   0
High-Level Keyword(s)_count    0
Abstract_word_count            0
Title_length                   0
dtype: int64
top topics:
Topics
Game Theory                                    16
Machine Learning (General/other)               12
Mechanism Design                               11
Classification                                 11
Natural Language Processing (General/Other)    11
Optimization                                    9
Planning (General/Other)                        9
Social Choice / Voting                          8
Data Mining and Knowledge Discovery             8

## 3. LOAD

In [21]:
# 3-1. Save as CSV (UTF-8)
out_csv = OUT_DIR / 'AAAI13_Papers_Clean.csv'
df_t.to_csv(out_csv, index=False, encoding='utf-8-sig')
print(f'saved: {out_csv}')

saved: C:\Users\USER\Desktop\[16-08-26] Cloudy\03.AAAI-13 Accepted Papers\output\AAAI13_Papers_Clean.csv


In [22]:
# 3-2. Save as Parquet (optional)
try:
    out_pq = OUT_DIR / 'AAAI13_Papers_Clean.parquet'
    df_t.to_parquet(out_pq, index=False)
    print(f'saved: {out_pq}')
except ImportError:
    print('pyarrow not installed - skip parquet')

saved: C:\Users\USER\Desktop\[16-08-26] Cloudy\03.AAAI-13 Accepted Papers\output\AAAI13_Papers_Clean.parquet


In [23]:
# 3-3. Verification: reload the saved file and compare
df_check = pd.read_csv(out_csv, encoding='utf-8-sig')
print(df_check.shape)
assert df_check.shape == df_t.shape, 'row count mismatch'
assert list(df_check.columns) == list(df_t.columns), 'column mismatch'
print('ETL verification passed')
df_check.head(2)

(150, 10)
ETL verification passed


,Title,Keywords,Topics,High-Level Keyword(s),Abstract,Keywords_count,Topics_count,High-Level Keyword(s)_count,Abstract_word_count,Title_length
0,The cascade auction – a mechanism for deterrin...,"Mediators, Auctions, Collusion, Ad Exchanges","Auctions and Market-Based Systems, E-Commerce,...",Multiagent Systems,We introduce a sealed bid auction of a single ...,4,4,1,69,69
1,Basis Adaptation for Sparse Nonlinear Reinforc...,"Reinforcement learning, Sparsity, Mirror desce...","Dimension Reduction/Feature Selection, Online ...","Machine Learning, Reasoning under Uncertainty",This paper presents a new approach to basis ad...,5,4,2,155,60


Authors:Carla Brodley